# Session 13 · Homework Solutions (Teacher Copy) — Advising the Fraud Team

**Machine Learning Foundations · Sanketana School of Code**

Worked solution with commentary — the module's flagship ethics deliverable. The policy threshold lands around **0.35**, reaching 80% recall (12/15 caught) for just **one** extra false alarm versus 0.5.

**Acceptable variation:** any threshold meeting the 80%-recall policy passes; a well-argued *higher* threshold that prioritises customers can also earn full marks **if** the student changes the stated policy and justifies it. The recommendation must cite numbers and name stakeholders on **both** sides.

## Step 1 · The confusion matrix at 0.5

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, precision_score, recall_score

fraud = pd.read_csv("../../../datasets/secondary/fraud_transactions.csv")
X = fraud.drop(columns=["is_fraud", "transaction_id"]).values
y = fraud["is_fraud"].values
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
scaler = StandardScaler().fit(X_train)
model = LogisticRegression(max_iter=1000).fit(scaler.transform(X_train), y_train)
proba = model.predict_proba(scaler.transform(X_test))[:, 1]

In [ ]:
def draw_matrix(y_true, pred, ax, title):
    """Confusion matrix, actual-fraud row first, cells named in plain language."""
    # labels=[1, 0] puts 'fraud' first on both axes, matching the board grid.
    cm = confusion_matrix(y_true, pred, labels=[1, 0])
    names = [["caught\n(TP)", "missed\n(FN)"],
             ["false alarm\n(FP)", "cleared\n(TN)"]]
    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{names[i][j]}\n{cm[i, j]}", ha="center", va="center",
                    fontsize=11, color="black")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["fraud", "legit"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["fraud", "legit"])
    ax.set_xlabel("PREDICTED"); ax.set_ylabel("ACTUAL")
    ax.set_title(title)

In [ ]:
pred_05 = (proba >= 0.5).astype(int)
fig, ax = plt.subplots(figsize=(5, 5))
draw_matrix(y_test, pred_05, ax, "Fraud model at threshold = 0.5")
plt.tight_layout(); plt.show()
print("at 0.5 -> recall", round(recall_score(y_test, pred_05), 2),
      "precision", round(precision_score(y_test, pred_05), 2))  # ~0.60 / ~0.90

## Step 2 · Meet the policy: catch at least 80% of fraud

In [ ]:
choice = None
for thr in np.round(np.arange(0.05, 0.95, 0.05), 2):
    if recall_score(y_test, (proba >= thr).astype(int)) >= 0.80:
        choice = thr   # highest threshold that still meets the policy
pred_choice = (proba >= choice).astype(int)
fig, ax = plt.subplots(figsize=(5, 5))
draw_matrix(y_test, pred_choice, ax, f"policy threshold = {choice}")
plt.tight_layout(); plt.show()
fp_choice = int(((pred_choice==1)&(y_test==0)).sum())
fp_05 = int(((pred_05==1)&(y_test==0)).sum())
print("policy threshold:", choice, " recall", round(recall_score(y_test, pred_choice), 2),
      "precision", round(precision_score(y_test, pred_choice), 2))   # ~0.80 / ~0.86
print("extra false alarms vs 0.5:", fp_choice - fp_05)   # ~1

## Step 3 · ✅ Model recommendation

*"I recommend a threshold of about **0.35**, giving **recall ≈ 0.80** and **precision ≈ 0.86** — we catch 12 of the 15 frauds while raising only about one more false alarm than the 0.5 default. In the grid: **caught (TP)** = a fraud stopped before the customer loses money; **missed (FN)** = a fraud that still slips through (3 remain — we are not perfect); **false alarm (FP)** = an honest customer's card frozen (only ~2 in this test set); **cleared (TN)** = the vast majority of honest transactions, untouched. **Who benefits:** customers who would otherwise be robbed, and the bank's fraud losses. **Who pays:** the handful of honest customers wrongly flagged, who face the inconvenience of a frozen card. The **customer-experience team's strongest objection** is that even a small false-alarm rate, at the bank's full transaction volume, means many thousands of upset customers per day — so they'd argue for a higher threshold and a fast un-freeze process. I'd pair my threshold with an easy one-tap 'this was me' confirmation to soften that cost."*

Full marks require: a stated threshold **with numbers**, the four cells mapped to consequences, stakeholders named on **both** sides, and the objection acknowledged. A one-sided or number-free recommendation does not pass.

**Module close:** this is exactly the reasoning the capstone's **Project Ethics Check (S22)** and Demo Day's *"what the model gets wrong and about whom"* (S24) will ask for. Evaluation is where the numbers meet the people — and choosing a threshold is choosing whom to protect.